# D8：Pandas 读取数据与选择行列（参考版）

先运行参考版，再在 `practice.ipynb` 从空白单元格独立写代码。
选择内核 **Python (D Learning)**。单元格是可以单独执行的一小段代码；
按 **Shift + Enter** 运行当前单元格并移到下一格。请从上往下运行。

A/B 都是虚构的教学数据。每行是一条学习记录，表头不计入数据行数。
`record_id` 是记录编号，`day` 是学习日序号，`topic` 是学习主题，
`minutes` 是学习分钟数。A 有 5 行 4 列，B 有 4 行 4 列。


In [ ]:
from pathlib import Path  # Path：用于表示和组合文件路径
import sys  # sys：查看当前 Python 解释器等运行信息
import pandas as pd  # pandas 简写成 pd，后面用 pd 调用它的功能

print("Python 路径：", sys.executable)
print("Pandas 版本：", pd.__version__)


## 1. 读取 CSV

CSV 是用逗号分隔字段的文本表格。`DataFrame` 表示二维表格，`df` 是常见的简写。
今天使用明确的 D 盘路径，避免当前工作文件夹不同导致找不到文件。
Notebook 中通常没有脚本的 `__file__` 变量，因此这里直接指定数据目录。


In [ ]:
data_dir = Path("D:/Learning/study/d8/data")  # 数据所在的文件夹
a = pd.read_csv(data_dir / "A.csv")  # 读取 A 表；/ 在这里表示拼接路径
b = pd.read_csv(data_dir / "B.csv")  # 读取 B 表，不会改变 A 表

print("A 表：")
print(a)
print("B 表：")
print(b)


## 2. 观察表格

`head()` 默认取前 5 行，`shape` 是（数据行数，列数），`info()` 打印各列的信息。
A 表恰好 5 行，因此 `a.head()` 会显示全部数据；使用 `a.head(3)` 观察前 3 行。
最左边默认的 0～4 是行索引，不属于 CSV 中的 4 个数据列。


In [ ]:
print(a.head())  # 默认显示前 5 行；这个样本只有 5 行
print(a.head(3))  # 显示前 3 行
print("A.shape：", a.shape)  # 预计 (5, 4)；shape 是属性，不加括号
print("B.shape：", b.shape)  # 预计 (4, 4)
print("前 3 行的形状：", a.head(3).shape)  # 预计 (3, 4)


In [ ]:
a.info()  # 直接调用；它打印信息并返回 None，不需要再套 print()


## 3. 选择一列和多列

`Series` 是一维的带标签数据；单个列名通常得到 Series。
用列名列表选取时保留 DataFrame，即使只选一列也是二维表格。


In [ ]:
minutes_series = a["minutes"]  # 一个列名：得到 Series
minutes_table = a[["minutes"]]  # 列名列表：得到只有一列的 DataFrame
selected_columns = a[["topic", "minutes"]]  # 选择两列，按给出的顺序排列

print(minutes_series)
print(type(minutes_series), minutes_series.shape)  # Series，(5,)
print(minutes_table)
print(type(minutes_table), minutes_table.shape)  # DataFrame，(5, 1)
print(selected_columns)
print("两列的形状：", selected_columns.shape)  # (5, 2)


## 4. 用 iloc 按位置选择

`iloc` 使用从 0 开始的整数位置：`[行位置, 列位置]`。
位置切片不包含终点，因此 `1:4` 对应位置 1、2、3。


In [ ]:
second_row = a.iloc[1]  # 第 2 行；单个行位置得到 Series
second_row_table = a.iloc[1:2]  # 第 2 行，保留表格形式
middle_rows = a.iloc[1:4]  # 第 2～4 行
chosen_cells = a.iloc[[0, 2], [2, 3]]  # 第 1、3 行的 topic 和 minutes 列

print(second_row)
print("单行 Series：", second_row.shape)  # (4,)，4 个字段
print("单行 DataFrame：", second_row_table.shape)  # (1, 4)
print("中间 3 行：", middle_rows.shape)  # (3, 4)
print(chosen_cells)
print("两行两列：", chosen_cells.shape)  # (2, 2)


## 5. 用不同的索引看清 loc 与 iloc

`loc` 按标签选择。默认索引是 0～4，`record_id` 此时仍是普通列。
创建一个新表，把 `record_id` 设为行索引；原表 a 不变。
新表的索引标签为 101～105，剩余列按顺序是 day、topic、minutes。


In [ ]:
indexed_a = a.set_index("record_id")  # 创建使用记录编号作索引的新表
print(indexed_a)
print("原表形状：", a.shape)  # (5, 4)
print("新表形状：", indexed_a.shape)  # (5, 3)，编号已经变成索引

print(indexed_a.loc[102])  # 标签为 102 的行，Series，(3,)
print(indexed_a.iloc[1])  # 位置为 1 的行，也就是第 2 行，与上一行相同
print(indexed_a.loc[[102], ["topic", "minutes"]])  # (1, 2)，保留表格
print(indexed_a.loc[102, "minutes"])  # 一个具体值：45，不是二维表格


## 6. 比较切片终点

在这份索引已排序且标签唯一的样本上，loc 的标签切片包含起点和终点，
iloc 的位置切片包含起点、不包含终点。注意新表已经没有 record_id 数据列。


In [ ]:
by_label = indexed_a.loc[101:103, ["topic", "minutes"]]  # 101、102、103
by_position = indexed_a.iloc[0:3, 1:3]  # 行位置 0、1、2；列位置 1、2
shorter = indexed_a.iloc[0:2, 1:3]  # 行位置只有 0、1

print(by_label)
print("标签切片形状：", by_label.shape)  # (3, 2)
print("对应的位置切片形状：", by_position.shape)  # (3, 2)
print("终点为 2 的位置切片：", shorter.shape)  # (2, 2)
print("前两种选法的结果相同：", by_label.equals(by_position))  # True


## 7. B 表练习参考

换一份样本，确认自己理解了行列数；今天暂时不合并 A/B 两张表。


In [ ]:
b_columns = b[["topic", "minutes"]]  # 预计 (4, 2)
b_first_two = b.iloc[:2]  # 位置 0、1，预计 (2, 4)
indexed_b = b.set_index("record_id")  # 预计 (4, 3)
b_cell = indexed_b.loc[202, "minutes"]  # 预计 40

print(b_columns)
print(b_columns.shape)
print(b_first_two)
print(b_first_two.shape)
print(indexed_b.shape)
print(b_cell)


## 8. 独立验收

关闭参考版，在 practice.ipynb 完成空白练习，并回答 notes.md。
1480 请从空白单元格重新写循环版本；每轮先累加再追加，循环结束后 return。
参考资料：[Pandas 读取数据](https://pandas.pydata.org/pandas-docs/version/2.3/getting_started/intro_tutorials/02_read_write.html)、
[选择行列](https://pandas.pydata.org/pandas-docs/version/2.3/getting_started/intro_tutorials/03_subset_data.html)。
